# 01 · Modelo de bigramas

**Peça nova:** o modelo de linguagem mais simples possível. Ele prevê o próximo caractere olhando **só para o caractere atual**, com uma tabela `vocab × vocab`: a linha `i` guarda as pontuações (logits) de cada caractere que pode vir depois de `i`.

Tudo o que vem depois (atenção, MLP, residual, normalização) é uma forma de dar ao modelo **mais contexto** e **mais capacidade** do que esta tabela. Por isso este notebook fixa a régua: a loss do bigrama é o número que todo o resto precisa bater.

**Roteiro de cada notebook**
1. Conceito e código da peça nova, escrito aqui mesmo (mexa à vontade).
2. Testes rápidos da peça.
3. Um treino de referência, comparado com o notebook anterior.
4. Experimentos: antes de cada um, escreva sua previsão na célula **Minha previsão**. Depois rode e compare.
5. Sugestão de entrada para o `DIARIO.md` do plano (código `F1-BIG`).

As execuções ficam em `runs/<nome>/` e podem ser comparadas entre notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2

import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from lab.data import get_batch, shakespeare
from lab.plot import compare, table
from lab.train import generate_text, pick_device, train

DEVICE = pick_device()  # "mps" no Mac. Para modelos minúsculos a CPU às vezes é mais rápida (ver experimento 4)
ds = shakespeare()
tok = ds.tok
print(ds, "| dispositivo:", DEVICE)

## Os dados

O tokenizador é por caractere: cada caractere distinto do texto vira um número de 0 a 64.

In [ ]:
print("vocabulário:", repr("".join(tok.chars)))
print(tok.encode("Hello!"), "->", repr(tok.decode(tok.encode("Hello!"))))
print()
print(tok.decode(ds.train[:300].tolist()))

Um batch sorteia trechos de `block_size` caracteres. O alvo `y` é o mesmo trecho deslocado de uma posição: em cada posição o modelo deve prever o caractere seguinte. Um trecho de 8 caracteres rende 8 exemplos de treino, com contextos de 1 a 8 caracteres.

In [ ]:
torch.manual_seed(0)
x, y = get_batch(ds.train, batch_size=2, block_size=8)
print("x =", x)
print("y =", y)
for t in range(8):
    print(f"contexto {tok.decode(x[0, :t + 1].tolist())!r:14} -> alvo {tok.decode([y[0, t].item()])!r}")

## A peça: `Bigram`

- `forward` devolve os logits `(B, T, V)` e, se houver alvos, a **cross-entropy**: a média de `-log p(alvo)`.
- `generate` sorteia um caractere de cada vez a partir das probabilidades da **última** posição e o acrescenta ao contexto.

A partir do notebook 02 essas duas funções vêm prontas de `lab.model.LanguageModel` (são as mesmas).

In [ ]:
class Bigram(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # linha i = logits do próximo caractere quando o atual é i
        self.table = nn.Embedding(vocab_size, vocab_size)
        self.block_size = None  # o bigrama não tem limite de contexto: só usa o último caractere

    def forward(self, idx, targets=None):
        logits = self.table(idx)  # (B, T, V)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)  # só a última posição importa
            nxt = torch.multinomial(probs, num_samples=1)              # sorteia o próximo caractere
            idx = torch.cat([idx, nxt], dim=1)
        return idx

### Teste: a loss inicial

Um modelo que não sabe nada deveria dar a mesma probabilidade, 1/65, a cada caractere. A loss seria `-ln(1/65) = ln(65) ≈ 4,17`. Confira:

In [ ]:
torch.manual_seed(1337)
model = Bigram(tok.vocab_size)
x, y = get_batch(ds.train, 32, 8)
_, loss = model(x, y)
print(f"loss inicial {loss.item():.3f}  |  ln({tok.vocab_size}) = {math.log(tok.vocab_size):.3f}")

with torch.no_grad():
    zerado = Bigram(tok.vocab_size)
    zerado.table.weight.zero_()  # logits todos iguais -> distribuição uniforme
    print(f"com a tabela zerada: {zerado(x, y)[1].item():.3f}")

A loss inicial é **maior** que 4,17 porque o `nn.Embedding` começa com valores sorteados de N(0, 1): o modelo começa "confiante e errado". Guarde isso: no notebook 07 o GPT inicializa os pesos com desvio 0,02 justamente para começar perto da previsão uniforme.

Texto gerado antes de treinar:

In [ ]:
print(generate_text(model, tok, max_new_tokens=200))

## Treino de referência

In [ ]:
torch.manual_seed(1337)
model = Bigram(tok.vocab_size)
run = train(model, ds, name="01-bigram", steps=3000, lr=1e-2, block_size=8, device=DEVICE)
print(run.sample)

## O teto do bigrama: contar em vez de aprender

Para um bigrama existe uma resposta exata: contar quantas vezes cada par `(atual, próximo)` aparece no treino e normalizar cada linha. Nenhum bigrama treinado por gradiente pode ser muito melhor do que isso na validação. Compare:

In [ ]:
V = tok.vocab_size
tr, va = ds.train.long(), ds.val.long()
counts = torch.zeros(V, V)
counts.index_put_((tr[:-1], tr[1:]), torch.ones(len(tr) - 1), accumulate=True)
P = (counts + 1) / (counts + 1).sum(dim=1, keepdim=True)  # +1: suavização, nenhum par fica com probabilidade zero
nll = -torch.log(P[va[:-1], va[1:]]).mean().item()
print(f"loss de validação do modelo de contagem: {nll:.4f}")
print(f"loss de validação do bigrama treinado:   {run.final['val_loss']:.4f}")

In [ ]:
learned = F.softmax(model.table.weight.detach().cpu(), dim=-1)
fig, axs = plt.subplots(1, 2, figsize=(13, 6))
labels = [repr(c)[1:-1] for c in tok.chars]
for ax, M, title in [(axs[0], P, "contagem"), (axs[1], learned, "aprendido")]:
    ax.imshow(M, cmap="Blues")
    ax.set_title(f"P(próximo | atual): {title}")
    ax.set_xticks(range(V), labels, fontsize=5)
    ax.set_yticks(range(V), labels, fontsize=5)
    ax.set_xlabel("próximo")
    ax.set_ylabel("atual")
plt.show()

---
## Experimentos

### 1. Learning rate
Treine o mesmo bigrama com learning rates de 1e-3 a 1.

**Minha previsão:** _(qual chega mais perto da loss de contagem em 2000 passos? alguma diverge?)_

In [ ]:
for lr in [1e-3, 1e-2, 1e-1, 1.0]:
    torch.manual_seed(1337)
    train(Bigram(V), ds, name=f"01-lr{lr:g}", steps=2000, lr=lr, block_size=8, device=DEVICE,
          verbose=False, sample_tokens=0)
compare(prefix="01-lr", title="Bigrama: learning rate")
table(prefix="01-lr")

### 2. Tamanho do contexto (`block_size`)
O bigrama só olha para o último caractere. O que acontece se cada trecho do batch tiver 8, 64 ou 256 caracteres?

**Minha previsão:** _(a loss final muda? a curva fica mais lisa ou mais ruidosa? o tempo por passo muda?)_

In [ ]:
for bs in [8, 64, 256]:
    torch.manual_seed(1337)
    train(Bigram(V), ds, name=f"01-ctx{bs}", steps=2000, lr=1e-2, block_size=bs, device=DEVICE,
          verbose=False, sample_tokens=0)
compare(prefix="01-ctx", title="Bigrama: block_size")
table(prefix="01-ctx")

### 3. Temperatura na geração
A temperatura divide os logits antes do softmax. Abaixo de 1 a distribuição fica mais "pontuda"; acima de 1, mais plana.

**Minha previsão:** _(como fica o texto com 0,3 e com 2,0?)_

In [ ]:
for temp in [0.3, 1.0, 2.0]:
    print(f"--- temperatura {temp} ---")
    print(generate_text(model, tok, max_new_tokens=200, temperature=temp), "\n")

### 4. CPU vs. GPU (MPS) para um modelo minúsculo
A GPU só compensa quando há conta suficiente para esconder o custo de mandar cada operação para ela.

**Minha previsão:** _(qual termina 500 passos primeiro?)_

In [ ]:
devices = ["cpu"] + ([DEVICE] if DEVICE != "cpu" else [])
for dev in devices:
    torch.manual_seed(1337)
    train(Bigram(V), ds, name=f"01-dev-{dev}", steps=500, lr=1e-2, block_size=8, device=dev,
          verbose=False, sample_tokens=0)
compare([f"01-dev-{d}" for d in devices], x="elapsed_s", title="Mesmo treino, dispositivos diferentes")
table(prefix="01-dev")

---
## Para o diário (`F1-BIG-01`)

- Qual foi a loss de validação do bigrama e quão perto ela ficou da contagem? Por que não dá para passar muito disso?
- O que o `block_size` mudou e o que **não** mudou? Por quê?
- Explique, sem consultar nada, por que a loss inicial era maior que ln(65).

**Próximo:** [02 · atenção](02_atencao.ipynb), onde cada caractere passa a olhar para os anteriores.